### Calculate the flexibility measures

In [5]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import os
import pyperclip

##### 0. Functions

In [23]:
def create_hapset(df):
    """
        Create a hapset from the cleaned df.
        Returns hapset: nxr np.array, 1 means homegame, 0 away
        also returns team_to_index, so we can link the haps to the teams        
    """
    
    teams = list(set(df["team_home"]) | set(df["team_away"]))
    n = len(teams)
    team_to_index = {team: i for i, team in enumerate(teams)}
    r = len(df["round"].unique())
    hapset = np.full((n, r), 2, dtype=int)
    min_round = df["round"].min()

    for _, row in df.iterrows():
        home_team = row["team_home"]
        away_team = row["team_away"]
        round_num = row["round"] - min_round
        hapset[team_to_index[home_team], round_num] = 1
        hapset[team_to_index[away_team], round_num] = 0
    
    if np.any(hapset == 2):
        print("Warning: Some matches are missing in the dataset.")
    
    return hapset, team_to_index

df = pd.read_csv(f"./cleaned_data/egypt_premier-league_2024-2025.csv")

hapset, team_to_index = create_hapset(df[(df["round"] >= 1) & (df["round"] <= 17)])
print(hapset)
print(team_to_index)

[[1 0 1 0 1 0 1 0 1 0 1 0 1 1 0 1 0]
 [1 0 1 0 1 1 0 1 0 1 0 1 0 1 0 1 0]
 [0 1 0 1 0 0 1 0 1 0 1 0 1 0 1 0 1]
 [1 0 1 0 1 1 0 0 1 0 1 0 0 1 0 1 0]
 [0 1 0 1 0 1 0 1 0 1 0 1 0 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0]
 [1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 0]
 [0 1 0 1 0 1 0 1 0 1 0 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 1]
 [0 1 0 1 0 1 1 0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 1 0 1 0 1 1 0 1 0 1 0 1 0]
 [1 0 1 0 1 0 0 1 0 1 0 1 1 0 1 0 1]
 [1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1]
 [1 0 1 0 1 0 1 0 0 1 0 1 0 1 0 1 0]
 [0 1 0 1 0 0 1 0 1 0 1 1 0 1 0 1 0]
 [0 1 0 1 0 1 0 1 1 0 1 0 1 0 1 0 1]
 [1 0 1 0 1 0 1 1 0 1 0 1 0 1 0 1 0]
 [1 0 1 0 0 1 0 1 0 0 1 0 1 0 1 0 1]]
{'Modern Sport': 0, 'El Gaish': 1, 'Enppi': 2, 'El Ismaily': 3, 'National Bank Egypt': 4, 'Smouha': 5, 'El Gouna': 6, 'Al Masry': 7, 'Al Ittihad': 8, 'ZED': 9, 'Ghazl El Mahallah': 10, 'Al Ahly': 11, 'Pharco': 12, 'Haras El Hodood': 13, 'Ceramica Cleopatra': 14, 'Petrojet': 15, 'Zamalek': 16, 'Pyramids': 17}


In [ ]:

# Flexibility measures
# Spread
def lambers_IP_checker(hapset, team1, team2, l): # briskorn + 1 match forcen
    n_teams = len(hapset)
    model = gp.Model("briskorn_condition")
    model.Params.OutputFlag = 0  # suppress Gurobi output
    
    # decision variabele = team i plays team j in round r
    x = {}
    for i in range(n_teams):
        for j in range(i+1, n_teams):
            for r in range(n_teams - 1):
                x[i,j,r] = model.addVar(vtype=GRB.BINARY, name=f"x_{i}_{j}_{r}")

    # Constraints
    # Each team plays once every round at most
    for i in range(n_teams):
        for p in range(n_teams - 1):
            # j < i
            sum_last= gp.quicksum(x[j, i, p] for j in range(n_teams) if j < i)
            
             # i < j
            sum_first = gp.quicksum(x[i, j, p] for j in range(n_teams) if j > i)

            model.addConstr(sum_first + sum_last == 1, name=f"Constr_One_Game_{i}_{p}")

    # You play every opponent at most once
    for i in range(n_teams):
        for j in range(i+1, n_teams):
            model.addConstr(
                gp.quicksum(x[i, j, p] for p in range(n_teams - 1)) == 1, 
                name=f"Constr_Pair_{i}_{j}"
            )
    
    # if a game between i and j is played in round r then their home/away status must match the hapset
    for i in range(n_teams):
        for j in range(i+1, n_teams):
            for r in range(n_teams - 1):
                if hapset[i][r] == hapset[j][r]:
                    model.addConstr(x[i,j,r] == 0, name=f"Constr_HAP_{i}_{j}_{r}")

    # force the match between team 1 and 2 to be played in round round
    if team1 > team2: team1, team2 = team2, team1 
    model.addConstr(x[team1, team2, l] == 1, name=f"Constr_Force_{team1}_{team2}_{l}")
    
    model.optimize()

    is_feasible = model.status == GRB.OPTIMAL
    schedule = []
    if is_feasible:
        schedule = [(i, j, r) for (i, j, r), var in x.items() if var.X > 0.5]
        
    model.dispose()

    return int(is_feasible), schedule

def spread_calculator(hapset):
    # Logic: calculate for each game(i,j) force to a specific round r) -> match(i,j,r)
    # Only count matches to the spread which were not found before.
    # only force matches which have not been seen before
    n_teams = len(hapset)
    matches_seen = set()
    total_spread = 0
    for i in range(n_teams):
        for j in range(i+1, n_teams):
            for r in range(n_teams - 1):
                # als zelfde home/away status of i and j in round r -> continue
                if hapset[i, r] == hapset[j, r] or (i,j,r) in matches_seen: continue
                feas, sched = lambers_IP_checker(hapset, i, j, r)
                if feas:
                    for match in sched:
                        if match not in matches_seen:
                            total_spread += 1
                            matches_seen.add(match)

    return total_spread


# Fixed Part
def lambers_IP_FP_SRR(hapset: np.ndarray, team1: int, team2: int):
    """
        Given a hapset and two teams it checks wether there can be two 
        schedules generated where team1 vs team2 is in different rounds.
        If this is the case the game is not fixed.
    """
    n_teams = hapset.shape[0]
    n_rounds = n_teams - 1
    W = 2

    if team1 > team2:
        team1, team2 = team2, team1

    model = gp.Model("FP_SRR")
    model.Params.OutputFlag = 0

    x = {}
    for w in range(W):
        for i in range(n_teams):
            for j in range(i + 1, n_teams):
                for r in range(n_rounds):
                    x[w, i, j, r] = model.addVar(vtype=GRB.BINARY, name=f"x_{w}_{i}_{j}_{r}")

    # Each match played exactly once per schedule
    for w in range(W):
        for i in range(n_teams):
            for j in range(i + 1, n_teams):
                model.addConstr(
                    gp.quicksum(x[w, i, j, r] for r in range(n_rounds)) == 1,
                    name=f"match_{w}_{i}_{j}"
                )

    # Each team plays exactly once per round per schedule
    for w in range(W):
        for i in range(n_teams):
            for r in range(n_rounds):
                model.addConstr(
                    gp.quicksum(x[w, i, j, r] for j in range(i + 1, n_teams))
                    + gp.quicksum(x[w, j, i, r] for j in range(i))
                    == 1,
                    name=f"one_game_{w}_{i}_{r}"
                )

    # HAP compatibility
    for w in range(W):
        for i in range(n_teams):
            for j in range(i + 1, n_teams):
                for r in range(n_rounds):
                    if hapset[i, r] == hapset[j, r]:
                        model.addConstr(x[w, i, j, r] == 0)

    # Target match {team1, team2} must be in different rounds across the two schedules
    for r in range(n_rounds):
        model.addConstr(
            gp.quicksum(x[w, team1, team2, r] for w in range(W)) <= 1,
            name=f"diff_round_{r}"
        )

    model.optimize()

    is_fixed = model.status != GRB.OPTIMAL
    model.dispose()
    return int(is_fixed)



def fixed_part_calculator(hapset):
    n_teams = hapset.shape[0]
    fp = 0
    for i in range(n_teams):
        for j in range(i + 1, n_teams):
                fp += lambers_IP_FP_SRR(hapset, i, j)
    print(f"Fixed part: {fp}/{int(n_teams * (n_teams-1)/2)}")
    return fp/(n_teams * (n_teams-1)/2)

# Width
def _schedule_feasible(h, W, time_limit=None):
    """
    Build and solve the feasibility IP (10)-(15) from Lambers et al. (2023)
    for a fixed number W of pairwise match-distinct schedules.
 
    OPTIMIZATION #1: only create x[w, i, j, r] when team i can plausibly
    play HOME against team j in round r under the HAP-set (i.e. h[i, r] == 1
    and h[j, r] == 0). This removes the need for constraints (12)-(13) and
    the "no self-match" constraints entirely, since those combinations are
    simply never represented as variables. All other constraints are
    rewritten to look up x.get((w, i, j, r), 0) instead of indexing x[...]
    directly, which is algebraically 0 for any combination that was never
    created.
 
    Parameters
    ----------
    h : np.ndarray, shape (num_teams, num_rounds)
        Binary HAP matrix; h[i, r] = 1 if team i plays home in round r.
    W : int
        Number of pairwise match-distinct schedules requested.
    time_limit : float, optional
        Gurobi time limit (seconds) for this IP.
 
    Returns
    -------
    bool
        True if W pairwise match-distinct schedules compatible with the
        HAP-set exist, False otherwise.
    """
    num_teams, num_rounds = h.shape
    teams = range(num_teams)
    rounds = range(num_rounds)
    schedules = range(W)
 
    model = gp.Model("hap_width")
    model.Params.OutputFlag = 0
    if time_limit is not None:
        model.Params.TimeLimit = time_limit
 
    # ------------------------------------------------------------------
    # Sparse variable creation: x[w, i, j, r] only exists if HAP-compatible
    # (i home i.e. h[i, r] == 1, j away i.e. h[j, r] == 0, i != j).
    # ------------------------------------------------------------------
    x = {}
    for r in rounds:
        home_teams = [i for i in teams if h[i, r] == 1]
        away_teams = [j for j in teams if h[j, r] == 0]
        for w in schedules:
            for i in home_teams:
                for j in away_teams:
                    if i != j:
                        x[w, i, j, r] = model.addVar(
                            vtype=GRB.BINARY, name=f"x_{w}_{i}_{j}_{r}"
                        )
    model.update()
 
    def xv(w, i, j, r):
        """Return the variable if it exists, else the constant 0."""
        return x.get((w, i, j, r), 0)
 
    # (10): each match {i, j} is played exactly once per schedule
    model.addConstrs(
        (
            gp.quicksum(xv(w, i, j, r) + xv(w, j, i, r) for r in rounds) == 1
            for w in schedules
            for i in teams
            for j in teams
            if i < j
        ),
        name="match_once",
    )
 
    # (11): each team plays exactly one match per round, per schedule
    model.addConstrs(
        (
            gp.quicksum(xv(w, i, j, r) + xv(w, j, i, r) for j in teams if j != i) == 1
            for w in schedules
            for i in teams
            for r in rounds
        ),
        name="one_game",
    )
 
    # (14): pairwise match-distinctness -- match {i, j} occupies at most one
    # round across ALL W schedules combined
    model.addConstrs(
        (
            gp.quicksum(xv(w, i, j, r) + xv(w, j, i, r) for w in schedules) <= 1
            for i in teams
            for j in teams
            if i < j
            for r in rounds
        ),
        name="distinct",
    )
 
    model.optimize()
 
    return model.Status == GRB.OPTIMAL
 
 
def hap_set_width(hap_set, hapset_feas=True, verbose=False):
    """
    Compute the width of a HAP-set (Definition 3.2 in Lambers, Goossens &
    Spieksma, 2023): the number of pairwise match-distinct schedules
    compatible with it.
 
    Parameters
    ----------
    hap_set : array-like, shape (2n, 2n-1)
        HAP-set. Entries can be 'H'/'A' strings or 1/0 integers.
        Row i, column r gives team i's home(1)/away(0) status in round r
        (r = 0, ..., 2n-2, i.e. round r+1 in the paper's 1-indexed notation).
 
    verbose : bool
        If True, print progress of the binary search.
 
    Returns
    -------
    int
        The width of the HAP-set (0 if the HAP-set itself is infeasible,
        i.e. not even a single compatible schedule exists).
    """
    num_teams, num_rounds = hap_set.shape
 
    if num_teams % 2 != 0:
        raise ValueError("A HAP-set must contain an even number of teams (2n).")
    if num_rounds != num_teams - 1:
        raise ValueError("A HAP-set of 2n teams must have 2n-1 rounds.")
 
    # First check basic feasibility (W = 1): does any compatible schedule exist?
    if not hapset_feas:
        if verbose:
            print("HAP-set is infeasible: width = 0")
        return 0
 
    w = 2
    while _schedule_feasible(hap_set, w):
        if verbose:
            print(f"Found feasible W = {w} ...")
        w += 1
    best = w - 1
 
    if verbose:
        print(f"width = {best}")
    return best


##### 1. Preprocessing

In [27]:
index = 54
files = os.listdir("./cleaned_data")
filename = files[index]
print(filename)

uruguay_liga-auf-uruguaya_2025_C.csv


In [8]:
index, rr, teams = 57, 2, 12
string = ""
for part in range(1, rr + 1):
    LB = (part - 1) * (teams - 1) + 1
    UB = part * (teams - 1)
    string += f", {index} : [{part}, {LB}, {UB}]"

print(string)
pyperclip.copy(string)

, 57 : [1, 1, 11], 57 : [2, 12, 22]


In [ ]:
# special cases: argentinia, australia, Canada, colombia, panama, Turkey, Uruguay intermedio and clausura, mls, 

for_loop_list = [
    [4, 1, 0, 11], [4, 2, 12, 22],
    [5, 1, 0, 15], [5, 2, 16, 30],
    [6, 1, 0, 19], [6, 2, 20, 39],
    [10, 1, 0, 9], [10, 2, 10, 18], [10, 3, 19, 27], [10, 4, 28, 36],
    [11, 1, 0, 13], [11, 2, 14, 26],
    [12, 1, 0, 15], [12, 2, 16, 30],
    [13, 1, 0, 11], [13, 2, 12, 22],
    [14, 1, 0, 15], [14, 2, 16, 30],
    [15, 1, 1, 17],
    [16, 1, 1, 19], [16, 2, 20, 38],
    [17, 1, 1, 17], [17, 2, 18, 34],
    [18, 1, 1, 17], [18, 2, 18, 34],
    [19, 1, 1, 11], [19, 2, 12, 22], [19, 3, 23, 33],
    [20, 1, 1, 15], [20, 2, 16, 30],
    [21, 1, 1, 9], [21, 2, 10, 18], [21, 3, 19, 27], [21, 4, 28, 36],
    [22, 1, 1, 19], [22, 2, 20, 38],
    [23, 1, 1, 15], [23, 2, 16, 30],
    [24, 1, 1, 19], [24, 2, 20, 38],
    [25, 1, 1, 15], [25, 2, 16, 30],
    [26, 1, 1, 11],
    [27, 1, 1, 11],
    [28, 1, 1, 17],
    [29, 1, 1, 17],
    [30, 1, 1, 15], [30, 2, 16, 30],
    [31, 1, 1, 17], [31, 2, 18, 34],
    [32, 1, 1, 19], [32, 2, 20, 38],
    [33, 1, 1, 11], [33, 2, 12, 22], [33, 3, 23, 33],
    [34, 1, 1, 15], [34, 2, 16, 30],
    [37, 1, 1, 11], [37, 2, 12, 22],
    [38, 1, 1, 11], [38, 2, 12, 22],
    [39, 1, 1, 17], [39, 2, 18, 34],
    [40, 1, 1, 17], [40, 2, 18, 34],
    [41, 1, 1, 15], [41, 2, 16, 30],
    [42, 1, 1, 15], [42, 2, 16, 30],
    [43, 1, 1, 11], [43, 2, 12, 22], [43, 3, 23, 33],
    [44, 1, 1, 15], [44, 2, 16, 30],
    [45, 1, 1, 15], [45, 2, 16, 30],
    [46, 1, 1, 11], [46, 2, 12, 22],
    [47, 1, 1, 11], [47, 2, 12, 22], [47, 3, 23, 33],
    [48, 1, 1, 19], [48, 2, 20, 38],
    [49, 1, 1, 15], [49, 2, 16, 30],
    [50, 1, 1, 11], [50, 2, 12, 22], [50, 3, 23, 33],
    [52, 1, 1, 15], [52, 2, 16, 30],
    [53, 1, 1, 15],
    [54, 1, 1, 15],
    [57, 1, 1, 11], [57, 2, 12, 22]
]

In [10]:
files = os.listdir("./cleaned_data")
pyperclip.copy(files)

##### 2. Compilation of Flex measures

In [28]:
files = ['algeria_ligue-1_2024-2025.csv', 'argentina_liga-profesional_2025_A.csv', 'argentina_liga-profesional_2025_C.csv', 'australia_a-league_2024-2025.csv', 'austria_bundesliga_2024-2025.csv', 'belgium_jupiler-pro-league_2024-2025.csv', 'brazil_serie-a-betano_2025.csv', 'canada_canadian-premier-league_2025.csv', 'colombia_primera-a_2025_A.csv', 'colombia_primera-a_2025_C.csv', 'croatia_hnl_2024-2025.csv', 'cyprus_cyprus-league_2024-2025.csv', 'czech-republic_chance-liga_2024-2025.csv', 'denmark_superliga_2024-2025.csv', 'ecuador_liga-pro_2025.csv', 'egypt_premier-league_2024-2025.csv', 'england_premier-league_2024-2025.csv', 'france_ligue-1_2024-2025.csv', 'germany_bundesliga_2024-2025.csv', 'hungary_nb-i_2024-2025.csv', 'iran_persian-gulf-pro-league_2024-2025.csv', 'ireland_premier-division_2025.csv', 'italy_serie-a_2024-2025.csv', 'ivory-coast_ligue-1_2024-2025.csv', 'japan_j1-league_2025.csv', 'luxembourg_bgl-ligue_2024-2025.csv', 'malta_premier-league_2024-2025_A.csv', 'malta_premier-league_2024-2025_C.csv', 'mexico_liga-mx_2024-2025_A.csv', 'mexico_liga-mx_2024-2025_C.csv', 'morocco_botola-pro_2024-2025.csv', 'netherlands_eredivisie_2024-2025.csv', 'nigeria_npfl_2024-2025.csv', 'northern-ireland_nifl-premiership_2024-2025.csv', 'norway_eliteserien_2025.csv', 'panama_lpf_2025_A.csv', 'panama_lpf_2025_C.csv', 'paraguay_copa-de-primera_2025_A.csv', 'paraguay_copa-de-primera_2025_C.csv', 'poland_ekstraklasa_2024-2025.csv', 'portugal_liga-portugal_2024-2025.csv', 'romania_superliga_2024-2025.csv', 'russia_premier-league_2024-2025.csv', 'scotland_premiership_2024-2025.csv', 'senegal_ligue-1_2024-2025.csv', 'serbia_mozzart-bet-super-liga_2024-2025.csv', 'slovakia_nike-liga_2024-2025.csv', 'south-korea_k-league-1_2025.csv', 'spain_laliga_2024-2025.csv', 'sweden_allsvenskan_2025.csv', 'switzerland_super-league_2024-2025.csv', 'turkey_super-lig_2024-2025.csv', 'ukraine_premier-league_2024-2025.csv', 'uruguay_liga-auf-uruguaya_2025_A.csv', 'uruguay_liga-auf-uruguaya_2025_C.csv', 'uruguay_liga-auf-uruguaya_2025_I.csv', 'usa_mls_2025.csv', 'wales_cymru-premier_2024-2025.csv']

for_loop_list = [[54, 1, 1, 15]]
for i, part, LB, UB in for_loop_list:
    index = i

    filename = files[index]
    try:
        df = pd.read_csv(f"./cleaned_data/{filename}")
        print(filename)
        country = filename.split("_")[0]
        suffix = filename.split("_")[-1].split(".")[0]
        if suffix in ["A", "C", "I"]:
            country += f"_{suffix}"
        # ADAPT
        part = part
        df_filtered = df[(df["round"] >= LB) & (df["round"] <= UB)]

        hapset = create_hapset(df_filtered)[0]
        #print(hapset)
        spread = spread_calculator(hapset)
        fixed_part = fixed_part_calculator(hapset)
        width = hap_set_width(hapset, hapset_feas=True, verbose=False)

        row  = {"country": country, "part": part, "hapset": hapset, "spread": spread, "fixed_part": fixed_part, "width": width}

        # Save results to CSV
        output_file = "flexibility_result.csv"
        header_needed = not os.path.isfile(output_file)
        df_row = pd.DataFrame([row])
        df_row.to_csv(output_file, mode='a', header=header_needed, index=False)
    except:
        print(f"Error processing {filename}. Skipping.")

uruguay_liga-auf-uruguaya_2025_C.csv
Fixed part: 4/120


##### 3. Special cases

In [53]:
# special cases: argentinia, australia, Canada, colombia, panama, Turkey, Uruguay intermedio and clausura, mls, 

# COLOMBIA: 1 extra ronde in SRR
fila_A, file_B = 'colombia_primera-a_2025_A.csv', 'colombia_primera-a_2025_C.csv'
df_A, df_B = pd.read_csv(f"./cleaned_data/{fila_A}"), pd.read_csv(f"./cleaned_data/{file_B}")
def get_opponent_schedule(df):
    """
        Constructs the opponent schedule as a 2D array nx(1+r),
        the first column is the team itself and then the next columns 
        are the opponents in each round.
    """
    teams = list(set(df["team_home"]) | set(df["team_away"]))
    n = len(teams)
    r = df["round"].max()
    team_to_index = {team: i for i, team in enumerate(teams)}

    opp_sched = np.zeros((n, r + 1), dtype=object)

    # Fill the first column with team names
    for team, index in team_to_index.items():
        opp_sched[index, 0] = team
    
    # Fill the opponent schedule based on the matches in the DataFrame
    for _, row in df.iterrows():
        home_team = row["team_home"]
        away_team = row["team_away"]
        round_num = row["round"]
        opp_sched[team_to_index[home_team], round_num] = away_team
        opp_sched[team_to_index[away_team], round_num] = home_team

    # Check if any team has an empty opponent slot (indicating missing data)
    if np.any(opp_sched == 0):
        print("Warning: Some matches are missing in the dataset.")

    return opp_sched




opp_sched_A = get_opponent_schedule(df_A)
opp_sched_B = get_opponent_schedule(df_B)


In [37]:
for row in opp_sched_A:
    print(row)
    break

['Fortaleza' 'Envigado' 'Once Caldas' 'Dep. Cali' 'Alianza' 'Aguilas'
 'Pereira' 'Atl. Nacional' 'Santa Fe' 'Deportes Tolima' 'Inter Bogotá'
 'America De Cali' 'Millonarios' 'Bucaramanga' 'Llaneros' 'Chico'
 'Ind. Medellin' 'Dep. Pasto' 'Inter Bogotá' 'Junior' 'U. Magdalena']


In [57]:
def get_duplicated_round(opp_sched):
    double_rounds = []
    for row in opp_sched:
        unique_opps, counts = np.unique(row, return_counts=True)
        double_opp = unique_opps[counts > 1][0]
        double_round = np.where(row == double_opp)[0][-1]
        print(double_round)
        double_rounds.append(double_round)
    return len(set(double_rounds)) == 1
get_duplicated_round(opp_sched_A)

18
18
19
10
17
10
10
19
10
17
17
16
17
10
10
10
10
16
18
18


False

In [48]:
opponents_only = opp_sched_A[0, 1:]
# Get unique values and how many times they appear
unique_opps, counts = np.unique(opponents_only, return_counts=True)
print(unique_opps)
print(counts)
print(unique_opps[counts > 1][0])

['Aguilas' 'Alianza' 'America De Cali' 'Atl. Nacional' 'Bucaramanga'
 'Chico' 'Dep. Cali' 'Dep. Pasto' 'Deportes Tolima' 'Envigado'
 'Ind. Medellin' 'Inter Bogotá' 'Junior' 'Llaneros' 'Millonarios'
 'Once Caldas' 'Pereira' 'Santa Fe' 'U. Magdalena']
[1 1 1 1 1 1 1 1 1 1 1 2 1 1 1 1 1 1 1]
Inter Bogotá
